# Student Performance — Notebook 4: Classification Models
**Project:** End-to-End AI Framework for Student Performance Analysis
**Student:** Megha Deopa | PRN: 2405020011520 | MBA AI/ML July 2024

**What is new here (mentor NEVER did this):**
- Pass/Fail CLASSIFICATION using Logistic Regression
- Confusion Matrix analysis
- Feature Importance chart (which factor matters most?)
- Learning Curve analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

# Classification
from sklearn.linear_model    import LogisticRegression, LinearRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier, RandomForestRegressor
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.metrics         import (accuracy_score, classification_report,
                                      confusion_matrix, r2_score)
from sklearn.preprocessing   import OneHotEncoder, StandardScaler
from sklearn.compose         import ColumnTransformer
from sklearn.model_selection import train_test_split, learning_curve

print('All libraries imported!')

In [ ]:
# Load and prepare data
df = pd.read_csv('data/stud.csv')
df['total score'] = df['math_score'] + df['reading_score'] + df['writing_score']
df['average']     = df['total score'] / 3
df['pass_fail']   = (df['average'] >= 40).astype(int)  # 1=Pass, 0=Fail

print('Dataset ready! Pass/Fail distribution:')
print(df['pass_fail'].value_counts())
print(f'\nPass Rate: {df["pass_fail"].mean()*100:.1f}%')

## 1. Preprocessing for Classification

In [ ]:
# Features and target
X_cls = df.drop(columns=['math_score','total score','average','pass_fail'])
y_cls = df['pass_fail']   # 1=Pass, 0=Fail
y_reg = df['math_score']  # for feature importance later

cat_features = X_cls.select_dtypes(include='object').columns
num_features = X_cls.select_dtypes(exclude='object').columns

preprocessor = ColumnTransformer([
    ('ohe',    OneHotEncoder(handle_unknown='ignore'), cat_features),
    ('scaler', StandardScaler(),                       num_features),
])

X_processed = preprocessor.fit_transform(X_cls)
print('Processed X shape:', X_processed.shape)

# Split
X_tr, X_te, y_cls_tr, y_cls_te = train_test_split(
    X_processed, y_cls, test_size=0.2, random_state=42)

_, _, y_reg_tr, y_reg_te = train_test_split(
    X_processed, y_reg, test_size=0.2, random_state=42)

print(f'Train: {X_tr.shape[0]} | Test: {X_te.shape[0]}')

## 2. Compare 4 Classification Models

In [ ]:
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree':        DecisionTreeClassifier(random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN Classifier':       KNeighborsClassifier(n_neighbors=5)
}

cls_results = []

print(f"{'Model':<25} {'Train Acc':>10} {'Test Acc':>10}")
print('=' * 50)

for name, clf in classifiers.items():
    clf.fit(X_tr, y_cls_tr)
    train_acc = accuracy_score(y_cls_tr, clf.predict(X_tr))
    test_acc  = accuracy_score(y_cls_te, clf.predict(X_te))
    cls_results.append({'Model':name,'Train Acc':train_acc,'Test Acc':test_acc})
    print(f'{name:<25} {train_acc:>10.4f} {test_acc:>10.4f}')

cls_df = pd.DataFrame(cls_results).sort_values('Test Acc', ascending=False)
print(f"\n Best Classifier: {cls_df.iloc[0]['Model']}")
print(f"   Test Accuracy:  {cls_df.iloc[0]['Test Acc']*100:.2f}%")

## 3. Confusion Matrix — Logistic Regression

In [ ]:
# Train best model
best_clf = LogisticRegression(max_iter=1000)
best_clf.fit(X_tr, y_cls_tr)
y_pred_cls = best_clf.predict(X_te)

cm = confusion_matrix(y_cls_te, y_pred_cls)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Fail','Predicted Pass'],
            yticklabels=['Actual Fail',   'Actual Pass'],
            linewidths=1, linecolor='white', annot_kws={'size':16})
plt.title('Confusion Matrix — Pass/Fail Prediction\n(Logistic Regression)', fontsize=14)
plt.tight_layout()
plt.show()

# Interpretation
tn, fp, fn, tp = cm.ravel()
print('Classification Report:')
print(classification_report(y_cls_te, y_pred_cls, target_names=['Fail','Pass']))

print('Plain English Interpretation:')
print(f'  Correctly predicted PASS : {tp} students')
print(f'  Correctly predicted FAIL : {tn} students')
print(f'  Predicted PASS but FAILED: {fp} students (False Positive — most dangerous error)')
print(f'  Predicted FAIL but PASSED: {fn} students (False Negative)')

## 4. Feature Importance — Which Factor Matters Most?

In [ ]:
# Use Random Forest for feature importance
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_tr, y_reg_tr)

# Get feature names after encoding
feature_names = preprocessor.get_feature_names_out()
importances   = rf_reg.feature_importances_

# Top 12 features
top_n   = 12
indices = np.argsort(importances)[::-1][:top_n]

clean_names = [
    feature_names[i].replace('ohe__','').replace('scaler__','')
    for i in indices[::-1]
]

plt.figure(figsize=(12, 7))
colors = ['#e74c3c' if i == top_n-1 else '#3498db' for i in range(top_n)]
plt.barh(range(top_n), importances[indices][::-1], color=colors, alpha=0.85)
plt.yticks(range(top_n), clean_names, fontsize=10)
plt.xlabel('Importance Score')
plt.title('Top 12 Features Influencing Math Score\n(Random Forest Feature Importance)', fontsize=14)
plt.tight_layout()
plt.show()

print('Top 5 Most Important Features:')
for rank, idx in enumerate(indices[:5], 1):
    clean = feature_names[idx].replace('ohe__','').replace('scaler__','')
    print(f'  {rank}. {clean}: {importances[idx]:.4f}')

## 5. Learning Curve — Does Model Need More Data?

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(
    LinearRegression(), X_processed, y_reg,
    cv=5, train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='r2', n_jobs=-1)

train_mean = np.mean(train_scores, axis=1)
train_std  = np.std(train_scores,  axis=1)
test_mean  = np.mean(test_scores,  axis=1)
test_std   = np.std(test_scores,   axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, 'o-', color='#3498db', label='Training Score',   linewidth=2)
plt.plot(train_sizes, test_mean,  's-', color='#2ecc71', label='Validation Score', linewidth=2)
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='#3498db')
plt.fill_between(train_sizes, test_mean  - test_std,  test_mean  + test_std,  alpha=0.1, color='#2ecc71')
plt.title('Learning Curve — Linear Regression', fontsize=14)
plt.xlabel('Number of Training Samples')
plt.ylabel('R² Score')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

gap = train_mean[-1] - test_mean[-1]
print(f'Final Training R²:   {train_mean[-1]:.4f}')
print(f'Final Validation R²: {test_mean[-1]:.4f}')
if gap < 0.05:   print('Gap is small — Model is well-fitted, no overfitting')
elif gap < 0.15: print('Moderate gap — Slight overfitting, acceptable')
else:            print('Large gap — Overfitting detected')